# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed (run once)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Use `@id` for referencing entities.

In [ ]:
# List all record sets and their @id
record_sets = metadata.record_set
print("Record Sets in the Dataset:")
for record_set in record_sets:
    print(f"  Record Set Name: {record_set.name} | Record Set @id: {record_set['@id']}")

# Explore fields in each record set
record_set_ids = []
for record_set in record_sets:
    record_set_ids.append(record_set['@id'])
    print(f"\nFields in Record Set '{record_set.name}' (@id={record_set['@id']}):")
    for field in record_set.field:
        print(f"  Field Name: {field.name} | Field @id: {field['@id']} | DataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `@id` field for referencing.

We'll extract data from each record set and store the DataFrames by their record set `@id`.

In [ ]:
# Extract all record sets by @id
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nLoaded DataFrame for Record Set @id: {rs_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. We'll choose a numeric field for analysis, filter values, normalize, and group by a categorical field.

**Remember:** Reference fields and columns **only by their `@id`**.

Let's select the first record set and demonstrate EDA using its fields.

In [ ]:
# Choose one record set based on previous overview (for demonstration)
target_record_set_id = record_set_ids[0]
df = dataframes[target_record_set_id]

# Find numeric fields from that record set
fields = [field for field in metadata.record_set[0].field if getattr(field, 'data_type', None) in ['Float', 'Integer', 'Number']]
print("Numeric fields available:")
for field in fields:
    print(f"  Field Name: {field.name} | Field @id: {field['@id']} | DataType: {field.data_type}")

# Select the first numeric field for EDA
if fields:
    numeric_field_id = fields[0]['@id']
    print(f"\nAnalyzing numeric field: {numeric_field_id}")
    # Use the @id to reference the column
    if numeric_field_id in df.columns:
        threshold = df[numeric_field_id].quantile(0.75) # Example: filter top 25% values
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head(3))

        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Find a group field (categorical/text)
        group_fields = [field for field in metadata.record_set[0].field if getattr(field, 'data_type', None) == 'Text']
        if group_fields:
            group_field_id = group_fields[0]['@id']
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"\nGrouped average of {numeric_field_id} by {group_field_id}:")
                print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and show grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if fields:
    # Histogram of numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, color="skyblue")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped data exists, plot
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,5))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id], palette="viridis")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We successfully loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County.

- Data was loaded using `mlcroissant`, referenced via Croissant schema `@id`.
- We identified record sets, fields, and extracted records.
- Basic EDA and visualizations revealed distributions and relationships in key indicators.
- The standardized Croissant schema structure enables scalable, reproducible data analyses with direct field referencing.

Further analyses can be extended by using the `@id` references to integrate fields and entities directly from the schema.